In [ ]:
import os

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

from agent.helpers.docker import container_logs, find_container

load_dotenv()

In [ ]:
@tool
async def docker_logs(container: str, tail: int = 50) -> str:
    """Fetch the last `tail` lines of a Docker container's stdout/stderr logs
    (read-only). `tail` is capped at 200."""
    tail = max(1, min(tail, 200))
    match = await find_container(container)
    if match is None:
        return f"(no container matching '{container}')"
    text = await container_logs(match["Id"], tail=tail)
    return text or f"(no logs for '{container}')"

In [31]:
llm = ChatOpenAI(
    model=os.environ["LLM_MODEL"],
    base_url=os.environ["LLM_BASE_URL"],
    api_key=os.environ.get("LLM_API_KEY"),
    temperature=0,
).bind_tools([docker_logs])

In [32]:
SYSTEM = "You are an infrastructure-operations assistant. Use the docker_logs tool when relevant, then answer concisely."

transcript: list = []


async def ask(question: str) -> str:
    """Ask one question. Makes at most one docker_logs call, then returns the model's answer."""
    messages = [SystemMessage(SYSTEM), HumanMessage(question)]
    reply = await llm.ainvoke(messages)
    messages.append(reply)
    if reply.tool_calls:
        call = reply.tool_calls[0]
        result = await docker_logs.ainvoke(call["args"] or {})
        messages.append(
            ToolMessage(tool_call_id=call["id"], content=str(result)[:4000])
        )
        reply = await llm.ainvoke(messages)
        messages.append(reply)
    transcript[:] = messages
    return reply.content

In [33]:
print(await ask("Show me the last log lines for the bifrost container."))



The `bifrost` container logs indicate it is running a Uvicorn server on port 8080 and is currently stuck in a **restart loop**.

Key observations:
*   The server starts up successfully (`Application startup complete`).
*   It handles a few requests (e.g., `GET /health`, `POST /v1/chat/completions`).
*   It then shuts down (`Shutting down`, `Application shutdown complete`).
*   Immediately after, it restarts (`Started server process [1]`, `Waiting for application startup`).
*   This cycle repeats continuously.

This behavior suggests the application is crashing or exiting immediately after starting, causing the container runtime to restart it repeatedly. You may want to check the application's exit code or look for more detailed error logs if available.


In [ ]:
def show_transcript(transcript: list) -> None:
    """Print each message with its tool calls, token usage, and reasoning."""
    for m in transcript:
        label = type(m).__name__
        body = m.content if not isinstance(m, ToolMessage) else f"[tool] {m.content}"
        if not body:
            body = "(tool call only)" if getattr(m, "tool_calls", None) else "(empty)"
        print(f"{label}: {body.replace(chr(10), chr(10) + '  ')}")

        reasoning = m.additional_kwargs.get("reasoning_content")
        if reasoning:
            print(f"  reasoning: {reasoning}")

        for call in getattr(m, "tool_calls", None) or []:
            print(f"  tool_call: {call['name']}({call['args']})")

        usage = getattr(m, "usage_metadata", None)
        if usage:
            print(f"  tokens: in={usage['input_tokens']} out={usage['output_tokens']} total={usage['total_tokens']}")

In [ ]:
show_transcript(transcript)